# Transformer: Parameter Counts

We consider a Transformer with:

- number of layers: $L$
- number of attention heads: $h$
- embedding size: $d$
- FFN hidden size: $d_{ff}$
- vocabulary size: $|V|$

We assume:

- sine/cosine positional encodings, so positional encodings have **0 learned parameters**
- standard Multi-Head Attention with learned $W^Q, W^K, W^V, W^O$
- standard FFN with two linear layers
- two LayerNorms per Transformer block
- token embedding is counted separately from the rest

## 1. Embedding Parameters

The token embedding matrix is:

$$
E \in \mathbb{R}^{|V| \times d}
$$

So the number of embedding parameters is:

$$
|V|d
$$

Since we are using sine/cosine positional encodings, the positional encodings contribute:

$$
0
$$

Therefore, total embedding-related parameters are:

$$
|V|d
$$

## 2. Parameters in One Transformer Layer

We map the parameter counts to the PreNorm computation graph equations.

### Multi-Head Attention

From the equations:

$$
Q^i = T^1 W^{Q_i}, \qquad K^i = T^1 W^{K_i}, \qquad V^i = T^1 W^{V_i}
$$

For each head $i$:

- $W^{Q_i} \in \mathbb{R}^{d \times d_k}$
- $W^{K_i} \in \mathbb{R}^{d \times d_k}$
- $W^{V_i} \in \mathbb{R}^{d \times d_k}$

where

$$
d_k = \frac{d}{h}
$$

Across all heads, the total number of parameters for $Q,K,V$ weights is:

$$
3hd\,d_k = 3d^2
$$

Including biases for all three projections adds:

$$
3d
$$

The output projection is:

$$
W^O \in \mathbb{R}^{d \times d}
$$

which contributes:

$$
d^2
$$

and its bias contributes:

$$
d
$$

So total Multi-Head Attention parameters per layer are:

$$
4d^2 + 4d
$$

## 3. Feedforward Network Parameters

From the equation:

$$
T^5 = \mathrm{FFN}(T^4)
$$

A standard FFN has the form:

$$
\mathrm{FFN}(x) = W_2\,\sigma(W_1x + b_1) + b_2
$$

where:

- $W_1 \in \mathbb{R}^{d \times d_{ff}}$
- $b_1 \in \mathbb{R}^{d_{ff}}$
- $W_2 \in \mathbb{R}^{d_{ff} \times d}$
- $b_2 \in \mathbb{R}^{d}$

Thus the FFN contributes:

$$
dd_{ff} + d_{ff}d + d_{ff} + d
$$

which simplifies to:

$$
2dd_{ff} + d_{ff} + d
$$

## 4. LayerNorm Parameters

Each LayerNorm has:

- scale parameter $\gamma \in \mathbb{R}^d$
- shift parameter $\beta \in \mathbb{R}^d$

So each LayerNorm contributes:

$$
2d
$$

There are 2 LayerNorms per Transformer layer, so LayerNorm contributes:

$$
4d
$$

## 5. Total Parameters Per Transformer Layer

Adding the contributions from:

- Multi-Head Attention: $4d^2 + 4d$
- FFN: $2dd_{ff} + d_{ff} + d$
- LayerNorm: $4d$

gives:

$$
(4d^2 + 4d) + (2dd_{ff} + d_{ff} + d) + 4d
$$

So the total number of parameters in one Transformer layer is:

$$
4d^2 + 2dd_{ff} + d_{ff} + 9d
$$

## 6. Total Parameters in the Full Transformer

For $L$ layers, the non-embedding parameters are:

$$
L\left(4d^2 + 2dd_{ff} + d_{ff} + 9d\right)
$$

The embedding parameters are:

$$
|V|d
$$

Therefore:

### Embedding parameters
$$
|V|d
$$

### Non-embedding parameters
$$
L\left(4d^2 + 2dd_{ff} + d_{ff} + 9d\right)
$$

### Total parameters
$$
|V|d + L\left(4d^2 + 2dd_{ff} + d_{ff} + 9d\right)
$$

## 7. Mapping to the Computation Graph Equations

Using the vectorized PreNorm equations:

$$
T^1 = \mathrm{LayerNorm}(X)
$$

This first LayerNorm contributes:

$$
2d
$$

For each head $i$:

$$
Q^i = T^1 W^{Q_i}, \qquad K^i = T^1 W^{K_i}, \qquad V^i = T^1 W^{V_i}
$$

These contribute:

$$
3d^2 + 3d
$$

Then:

$$
T^2 = (\mathrm{head}_1 \oplus \cdots \oplus \mathrm{head}_h)W^O
$$

This contributes:

$$
d^2 + d
$$

Then:

$$
T^4 = \mathrm{LayerNorm}(T^3)
$$

This second LayerNorm contributes:

$$
2d
$$

Then:

$$
T^5 = \mathrm{FFN}(T^4)
$$

This contributes:

$$
2dd_{ff} + d_{ff} + d
$$

So one full layer contributes:

$$
4d^2 + 2dd_{ff} + d_{ff} + 9d
$$

## Default Hyperparameters

Let

- number of layers: $L$
- number of attention heads: $h$
- embedding size: $d$
- FFN hidden size: $d_{ff}$
- vocabulary size: $|V|$

We assume sine/cosine positional encodings, so positional encodings have no learned parameters.

In [8]:
L = 6          # number of layers
h = 8          # number of attention heads
d = 512        # embedding size
d_ff = 2048    # FFN hidden size
V = 10000      # vocabulary size

embedding_params = V * d
non_embedding_params = L * (4 * d**2 + 2 * d * d_ff + d_ff + 9 * d)
total_params = embedding_params + non_embedding_params

print("Embedding parameters:", embedding_params)
print("Non-embedding parameters:", non_embedding_params)
print("Total parameters:", total_params)

Embedding parameters: 5120000
Non-embedding parameters: 18914304
Total parameters: 24034304


## 8. Conclusion

The Transformer has:

### Embedding parameters
$$
|V|d
$$

### Non-embedding parameters
$$
L\left(4d^2 + 2dd_{ff} + d_{ff} + 9d\right)
$$

### Total parameters
$$
|V|d + L\left(4d^2 + 2dd_{ff} + d_{ff} + 9d\right)
$$

These counts can be verified in PyTorch by summing the number of elements in all model parameters.
